# **Bayesian Networks and Autoregressive Language Models**

In [1]:
import random
from collections import defaultdict, Counter

random.seed(42)                      # reproducible results
START, END = "<START>", "<END>"

## **Part I: From Probability to Language**

**Question 1: Why is the chain-rule decomposition useful for generating text?**

The chain rule P(X1..XT) = P(X1) ∏ P(Xt | X1..Xt-1) breaks one huge joint distribution over whole sentences into a sequence of small next-token distributions. Text can then be generated one token at a time: sample X1, then X2 given X1, and so on. We never need to enumerate or store the probability of every possible sentence. We only need to estimate 'what comes next given what came before'. The decomposition is exact (no assumptions), so it is a valid way to model any sequence.

## **Part II: A Bayesian Network for Text**

**Question 2: What independence assumption does X1 → X2 → X3 → X4 make?**

Each word depends only on the immediately preceding word (first-order Markov assumption):

P(Xt | X1, ..., Xt-1) = P(Xt | Xt-1)

Equivalently, Xt is conditionally independent of X1..X(t-2) given Xt-1:  Xt ⊥ X1..X(t-2) | Xt-1.
So P(X1,X2,X3,X4) = P(X1)P(X2|X1)P(X3|X2)P(X4|X3).


## **Part III: Build a Small Language Dataset**

In [4]:
raw = """the cat sat on the mat
the cat sat on the rug
the dog sat on the mat
the dog ran to the park
the cat ran to the park
the dog sat on the rug"""

sentences = [line.lower().split() for line in raw.strip().split("\n")]

for s in sentences:
    print([START] + s + [END])

['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']


## **Part IV: Constructing the Conditional Probability Table**

**Question 3: Conditional Probability Distribution P(next word | current word)**

**P(next | the)** ("the" occurs 12 times)

| next | count | probability |
|---|---|---|
| cat | 3 | 3/12 = 0.25 |
| dog | 3 | 3/12 = 0.25 |
| mat | 2 | 2/12 ≈ 0.167 |
| rug | 2 | 2/12 ≈ 0.167 |
| park | 2 | 2/12 ≈ 0.167 |

**P(next | cat)** ("cat" occurs 3 times)

| next | probability |
|---|---|
| sat | 2/3 ≈ 0.667 |
| ran | 1/3 ≈ 0.333 |

**P(next | dog)** ("dog" occurs 3 times)

| next | probability |
|---|---|
| sat | 2/3 ≈ 0.667 |
| ran | 1/3 ≈ 0.333 |

**P(next | sat)** ("sat" occurs 4 times)

| next | probability |
|---|---|
| on | 4/4 = 1.0 |

**P(next | ran)** ("ran" occurs 2 times)

| next | probability |
|---|---|
| to | 2/2 = 1.0 |

Zero-probability transitions:

Any word never seen after the current word has probability 0

- after **the**: the, sat, ran, on, to, <END> (e.g. P(sat | the) = 0)
- after **cat**: the, cat, dog, on, to, mat, rug, park, <END> (e.g. P(mat | cat) = 0)
- after **dog**: the same as cat (e.g. P(park | dog) = 0)
- after **sat**: everything except "on" (e.g. P(the | sat) = 0)
- after **ran**: everything except "to" (e.g. P(sat | ran) = 0)

The model only assigns nonzero probability to bigrams seen in training. "sat" and "ran" are deterministic, since each has only one possible successor. Each row sums to 1.

## **Part V: Ask an LLM to Implement the Model**

**Prompt:**

Write a simple Python implementation of a first-order autoregressive language model.
The model should:
1. take a list of tokenised sentences as training data;
2. count transitions between consecutive tokens;
3. construct the conditional distribution P(Xt
| Xt−1);
4. display the probabilities for a specified previous token;
5. predict the most probable next token;
6. generate a sentence by repeatedly sampling the next token;
7. stop when the <END> token is generated.

Do not use a machine-learning library or a pretrained language model. Use ordinary
Python data structures and random sampling.

In [6]:
class FirstOrderLM:
    def __init__(self):
        self.counts = defaultdict(Counter)   # counts[prev][next] = C(prev, next)
        self.probs = {}                      # probs[prev][next]  = P(next | prev)

    def fit(self, sentences):
        for s in sentences:
            tokens = [START] + s + [END]
            for prev, nxt in zip(tokens, tokens[1:]):
                self.counts[prev][nxt] += 1
        # P(w_j | w_i) = C(w_i, w_j) / sum_k C(w_i, w_k)
        self.probs = {}
        for prev, c in self.counts.items():
            total = sum(c.values())
            self.probs[prev] = {nxt: n / total for nxt, n in c.items()}

    def dist(self, prev):
        return self.probs.get(prev, {})      # {} if prev never seen with a successor

    def show(self, prev):
        d = self.dist(prev)
        if not d:
            print(f"P(next | {prev}): no observed transitions")
            return
        print(f"P(next | {prev}):")
        for w, p in sorted(d.items(), key=lambda x: -x[1]):
            print(f"   {w:8s} {p:.3f}")

    def predict(self, prev):
        d = self.dist(prev)
        return max(d, key=d.get) if d else None

    def next_token(self, prev, mode="sample"):
        d = self.dist(prev)
        if not d:
            return None
        if mode == "greedy":
            return max(d, key=d.get)
        words, weights = zip(*d.items())
        return random.choices(words, weights=weights, k=1)[0]   # weighted sampling

    def generate(self, mode="sample", max_len=20):
        out, prev = [], START
        for _ in range(max_len):
            nxt = self.next_token(prev, mode)
            if nxt is None or nxt == END:
                break
            out.append(nxt)
            prev = nxt
        return " ".join(out)

model1 = FirstOrderLM()
model1.fit(sentences)

## **Part VI: Inspect the LLM-Generated Code**

**Question 4: Where are the transition counts stored?**
In `self.counts`, a dict of Counters: `counts[prev][next]` holds C(prev, next). They are filled inside `fit()`.

**Question 5: Where is P(Xt | Xt-1) computed?**
At the end of `fit()`: each count is divided by the row total, `n / sum(c.values())`, and stored in `self.probs[prev][next]`.

**Question 6: How does the program choose the next word?**
It depends on the mode in `next_token()`. In "greedy" mode it always takes the argmax. In "sample" mode it uses `random.choices` with the probabilities as weights. Greedy is deterministic: the same context always gives the same word. Sampling picks word w with probability P(w | prev), so less likely words still appear sometimes and the output varies.

**Question 7: What if a word has no observed transition?**
`dist()` returns an empty dict, `next_token()` returns None, and `generate()` stops. (It does not crash.) The model assigns no probability to the unseen context. Real systems handle this with smoothing or backoff.

## **Part VII: Test the Probability Model**


In [7]:
def check_normalisation(model, tol=1e-9):
    ok = True
    for ctx, d in model.probs.items():
        total = sum(d.values())
        print(ctx, round(total, 6))
        if abs(total - 1.0) > tol:
            ok = False
    print("\nALL DISTRIBUTIONS SUM TO 1" if ok else "\nNORMALISATION ERROR")

check_normalisation(model1)

<START> 1.0
the 1.0
cat 1.0
sat 1.0
on 1.0
mat 1.0
rug 1.0
dog 1.0
ran 1.0
to 1.0
park 1.0

ALL DISTRIBUTIONS SUM TO 1


**Question 8: What does a total of 0.87 mean?**

For every context, the probabilities of all possible next words must sum to 1. A total of 0.87 means 13% of the probability mass is missing, so the implementation is wrong. Likely causes: dividing by the wrong total (e.g. counting only some transitions, or excluding <END>), dropping some transitions, or rounding/filtering out entries. It is not a valid conditional distribution, and sampling from it would be biased.

## **Part VIII: Predicting the Next Word**


In [8]:
for w in ["the", "cat", "dog", "sat", "on", "ran", "to"]:
    model1.show(w)
    print(f"   argmax -> {model1.predict(w)}\n")

P(next | the):
   cat      0.250
   dog      0.250
   mat      0.167
   rug      0.167
   park     0.167
   argmax -> cat

P(next | cat):
   sat      0.667
   ran      0.333
   argmax -> sat

P(next | dog):
   sat      0.667
   ran      0.333
   argmax -> sat

P(next | sat):
   on       1.000
   argmax -> on

P(next | on):
   the      1.000
   argmax -> the

P(next | ran):
   to       1.000
   argmax -> to

P(next | to):
   the      1.000
   argmax -> the



**Question 9: Do the argmax predictions match human expectations?**

Often yes (sat → on, ran → to), but not always. For "the", cat and dog tie at 0.25, so argmax is just whichever the code sees first, which is an arbitrary tie-break. The model knows only the statistics of six sentences: it has no meaning or world knowledge, and it cannot predict words it never saw (e.g. "the bird"). A probability model reflects the frequencies in its training data, while human expectations come from grammar, meaning and experience, so they can differ.

## **Part IX: Generate Text**

In [9]:
generated = [model1.generate(mode="sample") for _ in range(20)]
for i, g in enumerate(generated, 1):
    print(i, g)

with open("generated_first_order.txt", "w") as f:
    f.write("\n".join(generated))

1 the cat sat on the dog ran to the cat sat on the cat sat on the dog ran to
2 the dog sat on the mat
3 the park
4 the dog sat on the rug
5 the park
6 the cat sat on the cat sat on the mat
7 the mat
8 the dog sat on the mat
9 the rug
10 the dog sat on the mat
11 the park
12 the mat
13 the mat
14 the mat
15 the mat
16 the rug
17 the cat sat on the cat sat on the park
18 the dog ran to the dog sat on the park
19 the rug
20 the park


## **Part X: Deterministic vs Probabilistic Generation**

In [10]:
print("GREEDY:")
for _ in range(5):
    print("  ", model1.generate(mode="greedy"))

print("\nSAMPLING:")
for _ in range(5):
    print("  ", model1.generate(mode="sample"))

GREEDY:
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on

SAMPLING:
   the dog sat on the dog sat on the cat ran to the mat
   the park
   the cat ran to the rug
   the park
   the rug


**Question 10: Which mode produces more variation?**

Sampling. Greedy always takes the argmax, so all five sentences are identical. In this model, greedy can also loop forever (the → cat → sat → on → the → cat → ...) until the `max_len` cutoff, because the argmax path never reaches <END>. Sampling draws from the full distribution, so different branches (cat/dog, sat/ran, mat/rug/park) appear in proportion to their probabilities, giving varied sentences and a chance to reach <END>.

## **Part XI: A Second-Order Bayesian Network**

**Question 11: First-order vs second-order**

1. Graph structure: first-order has one parent per node (Xt-1 → Xt). Second-order has two parents (Xt-2 → Xt ← Xt-1).
2. CPT: first-order is indexed by one previous word, giving V rows. Second-order is indexed by pairs, giving up to V² rows, each a distribution over V words.
3. Context: second-order sees two previous words, so it can distinguish e.g. "the cat" from "on the".
4. Data: it needs far more data, since there are many more contexts and each one is seen less often. With limited data many contexts are never observed.

## **Part XII: Use the LLM Again**

**Prompt:**

Modify the existing first-order autoregressive model into a second-order model.
The model should estimate
P(Xt
| Xt−2, Xt−1).
Represent the model using counts of observed triples and use these counts to construct
conditional probability distributions.
Do not replace the model with a neural network or a pretrained language model.

**What should change before accepting the code:** the context becomes a tuple (w_{t-2}, w_{t-1}); counts are over triples; the sequence is padded with two <START> tokens so that X1 ~ P(X1 | <START>, <START>); each row must still sum to 1.


In [11]:
class SecondOrderLM:
    def __init__(self):
        self.counts = defaultdict(Counter)   # counts[(a, b)][c] = C(a, b, c)
        self.probs = {}

    def fit(self, sentences):
        for s in sentences:
            tokens = [START, START] + s + [END]
            for a, b, c in zip(tokens, tokens[1:], tokens[2:]):
                self.counts[(a, b)][c] += 1
        self.probs = {}
        for ctx, cnt in self.counts.items():
            total = sum(cnt.values())
            self.probs[ctx] = {w: n / total for w, n in cnt.items()}

    def dist(self, ctx):
        return self.probs.get(ctx, {})

    def show(self, ctx):
        d = self.dist(ctx)
        if not d:
            print(f"P(next | {ctx}): context never observed")
            return
        print(f"P(next | {ctx}):")
        for w, p in sorted(d.items(), key=lambda x: -x[1]):
            print(f"   {w:8s} {p:.3f}")

    def predict(self, ctx):
        d = self.dist(ctx)
        return max(d, key=d.get) if d else None

    def next_token(self, ctx, mode="sample"):
        d = self.dist(ctx)
        if not d:
            return None
        if mode == "greedy":
            return max(d, key=d.get)
        words, weights = zip(*d.items())
        return random.choices(words, weights=weights, k=1)[0]

    def generate(self, mode="sample", max_len=20):
        out, ctx = [], (START, START)
        for _ in range(max_len):
            nxt = self.next_token(ctx, mode)
            if nxt is None or nxt == END:
                break
            out.append(nxt)
            ctx = (ctx[1], nxt)
        return " ".join(out)

model2 = SecondOrderLM()
model2.fit(sentences)

In [12]:
check_normalisation(model2)

print()
for ctx in [(START, START), (START, "the"), ("the", "cat"), ("the", "dog"),
            ("cat", "sat"), ("on", "the"), ("ran", "to"), ("to", "the")]:
    model2.show(ctx)
    print()

('<START>', '<START>') 1.0
('<START>', 'the') 1.0
('the', 'cat') 1.0
('cat', 'sat') 1.0
('sat', 'on') 1.0
('on', 'the') 1.0
('the', 'mat') 1.0
('the', 'rug') 1.0
('the', 'dog') 1.0
('dog', 'sat') 1.0
('dog', 'ran') 1.0
('ran', 'to') 1.0
('to', 'the') 1.0
('the', 'park') 1.0
('cat', 'ran') 1.0

ALL DISTRIBUTIONS SUM TO 1

P(next | ('<START>', '<START>')):
   the      1.000

P(next | ('<START>', 'the')):
   cat      0.500
   dog      0.500

P(next | ('the', 'cat')):
   sat      0.667
   ran      0.333

P(next | ('the', 'dog')):
   sat      0.667
   ran      0.333

P(next | ('cat', 'sat')):
   on       1.000

P(next | ('on', 'the')):
   mat      0.500
   rug      0.500

P(next | ('ran', 'to')):
   the      1.000

P(next | ('to', 'the')):
   park     1.000



In [13]:
generated2 = [model2.generate(mode="sample") for _ in range(20)]
for i, g in enumerate(generated2, 1):
    print(i, g)

with open("generated_second_order.txt", "w") as f:
    f.write("\n".join(generated2))

print("\nGREEDY (second-order):")
for _ in range(3):
    print("  ", model2.generate(mode="greedy"))

1 the cat ran to the park
2 the cat sat on the mat
3 the dog ran to the park
4 the dog sat on the rug
5 the cat sat on the rug
6 the cat sat on the mat
7 the cat sat on the rug
8 the dog ran to the park
9 the cat sat on the rug
10 the cat sat on the rug
11 the cat ran to the park
12 the dog ran to the park
13 the dog sat on the mat
14 the cat sat on the mat
15 the cat ran to the park
16 the dog ran to the park
17 the dog sat on the rug
18 the dog sat on the rug
19 the cat sat on the rug
20 the dog sat on the rug

GREEDY (second-order):
   the cat sat on the mat
   the cat sat on the mat
   the cat sat on the mat


## **Part XIII: Comparing the Two Models**

In [14]:
words = {w for s in sentences for w in s}
V_ctx = len(words | {START})          # possible context words (everything except END)
V_out = len(words | {END})            # possible next tokens

def n_params(m):                      # nonzero (context -> next) probabilities
    return sum(len(c) for c in m.counts.values())

train_set = {" ".join(s) for s in sentences}

def stats(name, model, gen_list, possible_contexts):
    observed = len(model.counts)
    print(f"--- {name} ---")
    print("observed contexts            :", observed)
    print("possible contexts            :", possible_contexts)
    print("zero-probability contexts    :", possible_contexts - observed)
    print("nonzero parameters           :", n_params(model))
    print("full CPT size (contexts x V) :", possible_contexts * V_out)
    print("distinct sentences (of 20)   :", len(set(gen_list)))
    print("novel (not in training) of 20:", sum(g not in train_set for g in gen_list))
    print()

stats("First-order", model1, generated, V_ctx)
stats("Second-order", model2, generated2, V_ctx ** 2)

print("Sample first-order :", generated[:5])
print("Sample second-order:", generated2[:5])

--- First-order ---
observed contexts            : 11
possible contexts            : 11
zero-probability contexts    : 0
nonzero parameters           : 17
full CPT size (contexts x V) : 121
distinct sentences (of 20)   : 9
novel (not in training) of 20: 16

--- Second-order ---
observed contexts            : 15
possible contexts            : 121
zero-probability contexts    : 106
nonzero parameters           : 19
full CPT size (contexts x V) : 1331
distinct sentences (of 20)   : 6
novel (not in training) of 20: 0

Sample first-order : ['the cat sat on the dog ran to the cat sat on the cat sat on the dog ran to', 'the dog sat on the mat', 'the park', 'the dog sat on the rug', 'the park']
Sample second-order: ['the cat ran to the park', 'the cat sat on the mat', 'the dog ran to the park', 'the dog sat on the rug', 'the cat sat on the rug']


In my run: first-order has 11 observed contexts and 17 nonzero parameters; second-order has 15 observed contexts out of 121 possible and 19 nonzero parameters.

- Parameters: second-order has more distinct parameters and a much larger possible table (V² x V vs V x V).
- Zero-probability contexts: far more in second-order (most of the 121 possible pairs are never observed).
- Diversity: first-order produces more varied and more novel sentences, including odd ones like "the cat ran to the mat". Second-order mostly reproduces training sentences.
- Coherence: second-order sentences are more grammatical/coherent because it sees two words of context, but it is closer to memorising the training data.

**Question 12: Why does more context help, yet make estimation harder?**

More context removes ambiguity: with two words the model can separate cases that one word conflates, so predictions are sharper and more coherent. But the CPT has V^k rows for context length k, so it grows exponentially. With a fixed amount of data, each context is observed few times (or never), so its probabilities are poorly estimated or zero. This is the bias-variance trade-off: more context lowers bias but raises variance and the data requirement.

## **Part XIV: The Connection to Modern Language Models**

Modern LLMs model the same P(Xt | X1..Xt-1), but a neural network estimates the distribution instead of a hand-built count table.

| | Simple BN model | Modern autoregressive LM |
|---|---|---|
| Representation | CPTs | Neural network |
| Context | Fixed window | Large learned context |
| Parameters | Explicit probabilities | Learned weights |
| Learning | Counting | Gradient-based training |
| Generation | Sampling | Sampling |

## **Part XV: Reflection on the Role of the LLM**

**Question 13: Why is Approach B preferable?**

- Specifying intended behaviour: Approach B states exactly what the program must compute, so the output can be judged against a specification instead of just "does it run".
- Understanding the representation: I must know the variables, dependencies and what the CPT stores, otherwise I cannot write the specification.
- Validating the implementation: I can check specific lines (where counts are stored, how probabilities are computed, how the next word is chosen).
- Testing probabilistic invariants: a clear model gives testable properties, e.g. every row sums to 1, no negative probabilities, generation only uses observed transitions.
- Implementation vs model: the model (P(Xt | Xt-1) from counts) is separate from the code. Approach A could silently return a different model (e.g. uniform random choice). Approach B lets me verify the code actually implements the intended model.

**What did the Bayesian network view add?**

1. A factorisation of the joint distribution: the graph tells us exactly how P(X1..XT) breaks into small conditional tables.
2. A way to reason about independence assumptions: first-order means Xt ⊥ X1..X(t-2) | Xt-1, and the second-order graph shows which assumption we relax.
3. A principled method for generation: ancestral sampling, i.e. sample each node given its parents in order.
4. A way to understand the effect of increasing context: adding a parent enlarges the CPT exponentially, which explains the data problem.
5. A way to test the implementation: the specification gives checkable properties, such as normalisation of each CPT row.